### Logreader 

In [31]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="gpt-oss:20b-cloud",
    temperature=0.5
)

Read log files 

In [32]:
# import os
# from langchain_core.tools import Tool

# @tool
# def summarise_logs() -> str:
#     "Read and return the summary of the first few lines from each log file in the logs directory."
#     log_directory = "./logs"
#     all_log = []
#     summary = ""
    
#     for filename in os.listdir(log_directory):
#         if filename.endswith(".log"):
#             with open(os.path.join(log_directory, filename), 'r') as file:
#                 log_content = file.read()
#                 prompt = f"Summarize the following log content:\n{log_content}\nSummary:"
#                 response = llm.invoke({"messages": [{"role": "user", "content": prompt}]})
#                 summary += f"Summary of {filename}:\n{response['output']}\n\n"
    
#     return summary

In [33]:
 ##I am using Visual Studio Code and I am trying to write the code for reading the log files. Here is my code:


import os
from langchain.tools import tool
from langchain_community.tools import tool
from langchain_core.messages import HumanMessage


@tool
def summarize_logs() -> str:
    "Read and return the summary of the first few lines from each log file in the logs directory."
    log_directory = "./logs"
    all_logs = []
    
    for filename in os.listdir(log_directory):
        with open(os.path.join(log_directory, filename)) as f:
            all_logs.extend(f"{filename}:{line.strip()}" for line in f.readlines()[:200])
    return "\n".join(all_logs) or "No log files found."



In [34]:
import os
from langchain.agents import create_agent

@tool
def add_numbers(a: int, b: int) -> int:
    """Adds two numbers together."""
    return a + b   
@tool
def multiply_numbers(a: int, b: int) -> int:
    """Multiplies two numbers together."""
    return a * b
@tool
def subtract_numbers(a: int, b: int) -> int:
    """Subtracts the second number from the first."""
    return a - b
tools = [add_numbers, multiply_numbers, subtract_numbers, summarize_logs]
agent = create_agent(
    tools=tools,
    model=llm  
)



In [35]:
query = "what Are the errors in my logs? Please summarise them."
response = agent.invoke({"messages": [HumanMessage(content=query)]})
print("Final Result:", response["messages"][-1].content)

Final Result: **Summary of error‑level events (timestamp – component – brief description)**  

| Time | Source | Error |
|------|--------|-------|
| 10:00:03 | **DBConnection** | Connection pool exhausted – retrying |
| 10:00:12 | **PaymentGateway** | 504 Gateway Timeout (transaction TXN_88292 failed) |
| 10:00:24 | **Middleware** | Invalid JWT token signature – authentication failure |
| 10:01:09 | **Frontend** | JavaScript exception: “undefined is not a function” |
| 10:01:30 | **CDN** | Asset not found: `/img/logo_v2.png` |
| 10:01:43 | **Analytics** | Event dropped due to schema mismatch |

---

**System‑level errors (kernel / service layer)**  

| Time | Source | Error |
|------|--------|-------|
| 10:00:21 | **kernel** | Out‑of‑memory: killed process 2201 (python3) |
| 10:00:23 | **systemd** | `service-monitor.service` main process exited (killed, status 9/KILL) |
| 10:00:24 | **systemd** | `service-monitor.service` failed (signal) |
| 10:00:31 | **kernel** | Possible SYN floodin